# 04a - Prophet Forecasting

**Objective**: Forecast maize prices using Facebook Prophet

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✓ Libraries imported')

In [ ]:
# Load datatry:    df = pd.read_csv('../data/clean/maize_features.csv')except FileNotFoundError:    df = pd.read_csv('data/clean/maize_features.csv')df_prophet = df[['date', 'price']].rename(columns={'date': 'ds', 'price': 'y'})print(f'Dataset: {df_prophet.shape}')df_prophet.head()

In [ ]:
# Train-test split
train_size = len(df_prophet) - 12
train = df_prophet[:train_size]
test = df_prophet[train_size:]
print(f'Train: {len(train)}, Test: {len(test)}')

In [ ]:
# Train Prophet model
model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model.fit(train)
print('✓ Model trained')

In [ ]:
# Make predictions
future = model.make_future_dataframe(periods=12, freq='MS')
forecast = model.predict(future)
print('✓ Forecast generated')
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

In [ ]:
# Evaluate on test set
test_forecast = forecast[forecast['ds'].isin(test['ds'])]
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test['y'], test_forecast['yhat'])
rmse = np.sqrt(mean_squared_error(test['y'], test_forecast['yhat']))
mape = np.mean(np.abs((test['y'] - test_forecast['yhat']) / test['y'])) * 100
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')

In [ ]:
# Save model
import joblib
joblib.dump(model, '../models/trained/prophet_maize_model.pkl')
forecast.to_csv('../models/trained/prophet_forecast.csv', index=False)
print('✓ Model saved')